In [21]:
from pathlib import Path
from math import inf
import heapq
import openpyxl

ARCHIVO_EXCEL=Path("Datos_SucursalesSimán.xlsx")

#convierte el valor del excel en float
def numero(valor):
    if valor is None:
        return None
    
    try:
        
        return float(valor)
    
    except(TypeError,ValueError):
        return None
    
    
def formato_km(valor):
    if valor is None or valor==inf:
        return "N/D"
    
    return f"{valor:.2f} km"

CANDIDATOS={
    
    "A": {
        "nombre": "Candidato A",
        "zona": "Zona Norte",
        "ubicacion": "Sector La 105 Brigada",
    },
    "B": {
        "nombre": "Candidato B",
        "zona": "Zona Sur",
        "ubicacion": "Zona industrial, Sector el Polvorín",
    },
    "C": {
        "nombre": "Candidato C",
        "zona": "Zona Centro",
        "ubicacion": "Barrio Los Andes, 21102 San Pedro Sula, Cortés, Honduras",
    },
    
}


def nombre_candidato(letra):
    
    return CANDIDATOS[str(letra).upper()]["nombre"]

def datos_candidato(letra):
    return CANDIDATOS[str(letra).upper()]

if not ARCHIVO_EXCEL.exists():
    
    raise FileNotFoundError(f"No se encontro el archivo: {ARCHIVO_EXCEL.resolve()}")

wb=openpyxl.load_workbook(ARCHIVO_EXCEL,data_only=True)


#leer los 5 bloques, Lee la hoja 'Bloques de Sucursales' dentro del excel
def cargar_bloques(hoja):
    
    bloques={"Bloque 1":[],"Bloque 2":[],"Bloque 3":[],"Bloque 4":[],"Bloque 5":[],}
    
    #bloques 1,2,3
    for fila in range(3,15):
        columnas=[(2,"Bloque 1"),(5,"Bloque 2"),(8,"Bloque 3"),]
        
        for columna_nombre,bloque in columnas:
            nombre=hoja.cell(fila,columna_nombre).value
            if nombre:
                bloques[bloque].append(str(nombre).strip())
                
    #bloque 4,5
    for fila in range(19,31):
        columnas=[(2,"Bloque 4"),(5,"Bloque 5")]
        
        for columna_nombre,bloque in columnas:
                    nombre=hoja.cell(fila,columna_nombre).value
                    if nombre:
                        bloques[bloque].append(str(nombre).strip())
                        
    return bloques
        
        
hoja_bloques=wb["Bloques de Sucursales"]
bloques=cargar_bloques(hoja_bloques)

#verificar q cada bloque debe tener 12 farmacias 
for bloque,farmacias in bloques.items():
    
    if len(farmacias)!=12:
        raise ValueError(f"{bloque} tiene {len(farmacias)} farmacias; se esperan 12")
    

#Aca se construyen el grafo de las distancias vecinas 
#Se guarda como conexion no dirigida porque el peso de Dijkstra sera el promedio ida/vuelta. Esto permite que todas las farmacias del bloque sean alcanzables.    

grafo={}

def agregar_vertice(nombre):
    if nombre not in grafo:
        grafo[nombre]={}
    
#Agrega una conexion con pesos ida, vuelta y promedio    
def agregar_conexion(a,b,ida,vuelta):
    
    ida=numero(ida)
    vuelta=numero(vuelta)
    
    if ida is None or vuelta is None:
        return
    
    promedio=(ida+vuelta)/2
    agregar_vertice(a)
    agregar_vertice(b)
    
    datos={"ida":ida,"vuelta":vuelta,"peso":promedio}
    
    #si la misma conexion apareciera mas de una vez, conservarla de menor promedio.
    anterior=grafo[a].get(b)
    if anterior is None or promedio<anterior["peso"]:
        
        grafo[a][b]=datos
        grafo[b][a]={"ida":ida,"vuelta":vuelta,"peso":promedio}
        
        
hojas_vecinos=wb["Distancias de Farmacias Vecinas"]
farmacia_actual=None

for fila in range(2,hojas_vecinos.max_row+1):
    numero_farmacia=hojas_vecinos.cell(fila,1).value
    nombre=hojas_vecinos.cell(fila,2).value
    
    #una fila con numero de farmacia marca el inicio de un grupo
    if numero_farmacia is not None and nombre:
        farmacia_actual=str(nombre).strip()
        agregar_vertice(farmacia_actual)
        continue
    
    #las siguientes filas contienen sus 3 farmacias vecinas
    if farmacia_actual and nombre:
        vecino=str(nombre).strip()
        ida=hojas_vecinos.cell(fila,4).value
        vuelta=hojas_vecinos.cell(fila,5).value
        
        agregar_conexion(farmacia_actual,vecino,ida,vuelta)
        
        
#leer y agregar los 3 candidatos 

Candidato_distancias={"A":{},"B":{},"C":{}}
hoja_candidatos=wb["Distancias de Candidatos"]

candidato_actual=None

for fila in range(2,17):
    letra=hoja_candidatos.cell(fila,1).value
    bloque=hoja_candidatos.cell(fila,2).value
    sucursal=hoja_candidatos.cell(fila,3).value
    ida=hoja_candidatos.cell(fila,4).value
    vuelta=hoja_candidatos.cell(fila,5).value
    
    if letra:
        candidato_actual=str(letra).strip().upper()
        
    if candidato_actual not in Candidato_distancias:
        continue
    
    if bloque is None or sucursal is None:
        continue
    
    ida=numero(ida)
    vuelta=numero(vuelta)
    
    if ida is None or vuelta is None:
        continue
    
    bloque=int(bloque)
    sucursal=str(sucursal).strip()
    Candidato_distancias[candidato_actual][bloque]={"sucursal":sucursal,"ida":ida,"vuelta":vuelta,"promedio":(ida+vuelta)/2}
    
#Aqui se agregan los candidatos como vertices y conectarlos con la entrada de cada bloque
for letra,datos_bloques in Candidato_distancias.items():
    candidato=nombre_candidato(letra)
    agregar_vertice(candidato)
    
    for numero_bloque,datos in datos_bloques.items():
        entrada=datos["sucursal"]
        agregar_conexion(candidato,entrada,datos["ida"],datos["vuelta"])
        
        
#5 dijkstra
#Dijkstra usando como peso:  peso = (ida + vuelta) / 2  
#Devuelve: distancias -> distancia minima promedio   anteriores -> permite reconstruir el camino
def dijkstra(grafo_local,inicio):

    distancias={vertice:inf for vertice in grafo_local}
    anteriores={vertice:None for vertice in grafo_local}
    
    distancias[inicio]=0.0
    cola=[(0.0,inicio)]
    
    while cola:
        distancia_actual,actual=heapq.heappop(cola)
        
        if distancia_actual>distancias[actual]:
            continue
        
        for vecino,datos in grafo_local[actual].items():
            peso=datos["peso"]
            nueva_distancia=distancia_actual+peso
            
            if nueva_distancia<distancias[vecino]:
                distancias[vecino]=nueva_distancia
                anteriores[vecino]=actual
                heapq.heappush(cola,(nueva_distancia,vecino))
                
    return distancias,anteriores


#6 reconstruir la ruta 

def reconstruir_ruta(anteriores,inicio,destino):
    ruta=[]
    actual=destino
    
    while actual is not None:
        ruta.append(actual)
        if actual==inicio:
            break
        actual=anteriores.get(actual)
        
    if not ruta or ruta[-1]!=inicio:
        return[]
    
    ruta.reverse()
    return ruta

#calcular ida y vuelta sobre la ruta de dijkstra
#La ruta elegida por Dijkstra usa el promedio de cada conexion.
#Despues se suman por separado las distancias de ida y vuelta.

def calcular_ida_vuelta(ruta):
    total_ida=0.0
    total_vuelta=0.0
    
    for actual,siguiente in zip(ruta,ruta[1:]):
        datos=grafo[actual][siguiente]
        total_ida+=datos["ida"]
        total_vuelta+=datos["vuelta"]
        
    promedio=(total_ida+total_vuelta)/2
    
    return total_ida,total_vuelta,promedio


#Aqui se crea cada subgrafo de cada bloque
#Crea el grafo correspondiente a un bloque.
#Incluye:  - Las 12 farmacias del bloque. 
# - El candidato.
# - La conexiin del candidato con la entrada del bloque.

def crear_subgrafo_bloque(bloque,candidato):
    nodos=set(bloques[bloque])
    nodos.add(candidato)
    subgrafo={nodo:{}for nodo in nodos}
    
    for u in nodos:
        for v,datos in grafo.get(u,{}).items():
            if v in nodos:
                subgrafo[u][v]=datos
                
    return subgrafo


#Aqui se analiza candidato + bloque
def analizar_bloque(letra_candidato,numero_bloque):
    candidato=nombre_candidato(letra_candidato)
    bloque=f"Bloque {numero_bloque}"
    
    subgrafo=crear_subgrafo_bloque(bloque,candidato)
    distancias,anteriores=dijkstra(subgrafo,candidato)
    resultados=[]
    
    for farmacia in bloques[bloque]:
        distancia_dijkstra=distancias.get(farmacia,inf)
        
        if distancia_dijkstra==inf:
            resultados.append({
                "farmacia": farmacia,
                "dijkstra": inf,
                "ida": inf,"vuelta": inf,"promedio": inf,"ruta": [],})
            continue
        
        ruta=reconstruir_ruta(anteriores,candidato,farmacia)
        ida,vuelta,promedio=calcular_ida_vuelta(ruta)
        resultados.append({
                        "farmacia": farmacia,
                        "dijkstra": distancia_dijkstra,
                        "ida": ida,"vuelta": vuelta,"promedio": promedio,
                        "ruta": ruta,})
        
    resultados.sort(key=lambda x:x["promedio"])
    return resultados


#10 mostrar resultados
def imprimir_linea():# XD
    print("="*90)
    
def imprimir_resultado_bloque(letra_candidato, numero_bloque, resultados):
    info=datos_candidato(letra_candidato)
    candidato=info["nombre"]
    bloque=f"Bloque {numero_bloque}"

    imprimir_linea()
    print(f"{candidato.upper()} - {info['zona'].upper()}")
    print(f"Ubicacion: {info['ubicacion']}")
    print(f"Análisis: {bloque.upper()}")
    imprimir_linea()

    print("\nLAS 12 FARMACIAS ORDENADAS DE MENOR A MAYOR DISTANCIA\n")

    for posicion, resultado in enumerate(resultados, start=1):
        print(f"{posicion:2}. {resultado['farmacia']}")
        print(f"    Dijkstra (promedio): {formato_km(resultado['dijkstra'])}")
        print(f"    Ida:                 {formato_km(resultado['ida'])}")
        print(f"    Vuelta:              {formato_km(resultado['vuelta'])}")
        print(f"    Promedio ida/vuelta: {formato_km(resultado['promedio'])}")

        if resultado["ruta"]:
            print("    Ruta:")
            print("      " + " -> ".join(resultado["ruta"]))
        else:
            print("    Ruta: NO DISPONIBLE")

        print()

    imprimir_linea()
    print("LAS 2 FARMACIAS MÁS CERCANAS")
    imprimir_linea()

    for posicion, resultado in enumerate(resultados[:2], start=1):
        print(f"{posicion}. {resultado['farmacia']}")
        print(f"   Promedio ida/vuelta: {formato_km(resultado['promedio'])}")
        print(f"   Ida:                 {formato_km(resultado['ida'])}")
        print(f"   Vuelta:              {formato_km(resultado['vuelta'])}")



# ------------------------------------------------------------
# 11. EJECUCION PRINCIPAL
# ------------------------------------------------------------
print("\n")
imprimir_linea()
print("       PROYECTO DIJKSTRA - FARMACIAS SIMAN SPS")
imprimir_linea()
print("\nDatos cargados directamente desde:")
print(f"{ARCHIVO_EXCEL.resolve()}\n")

print("ESTRUCTURA DEL PROYECTO")
print("-" * 90)
print("Cantidad de farmacias por bloque:")
for bloque, farmacias in bloques.items():
    print(f"  {bloque}: {len(farmacias)} farmacias")

print("\nCANDIDATOS")
print("-" * 90)
for letra in ["A", "B", "C"]:
    info = datos_candidato(letra)
    print(f"{info['nombre']} - {info['zona']}")
    print(f"  Ubicación: {info['ubicacion']}")

print("\nMetodo de cálculo:")
print("  • Dijkstra utiliza el promedio de ida/vuelta como peso de cada conexion.")
print("  • La ruta se obtiene unicamente mediante las conexiones del grafo.")
print("  • Luego se calculan por separado la distancia de ida, vuelta y promedio.")

# Guardaremos los resultados generales para el resumen.
resumen = {"A": {}, "B": {}, "C": {}}

for letra in ["A", "B", "C"]:
    for numero_bloque in range(1, 6):
        resultados = analizar_bloque(letra, numero_bloque)
        resumen[letra][numero_bloque] = resultados
        imprimir_resultado_bloque(letra, numero_bloque, resultados)


# ------------------------------------------------------------
# 12. RESUMEN GENERAL
# ------------------------------------------------------------
imprimir_linea()
print("RESUMEN GENERAL")
imprimir_linea()

for letra in ["A", "B", "C"]:
    info = datos_candidato(letra)
    print(f"\n{info['nombre'].upper()} - {info['zona'].upper()}")
    print(f"Ubicacion: {info['ubicacion']}")
    print("-" * 90)

    for numero_bloque in range(1, 6):
        resultados = resumen[letra][numero_bloque]
        validos = [r for r in resultados if r["promedio"] != inf]

        if not validos:
            print(f"Bloque {numero_bloque}: sin ruta disponible")
            continue

        mas_cercana = validos[0]

        print(
            f"Bloque {numero_bloque}: "
            f"{mas_cercana['farmacia']} | "
            f"promedio = {formato_km(mas_cercana['promedio'])} | "
            f"ida = {formato_km(mas_cercana['ida'])} | "
            f"vuelta = {formato_km(mas_cercana['vuelta'])}"
        )

print("\n")
imprimir_linea()
print("FIN DEL ANÁLISIS")
imprimir_linea()






       PROYECTO DIJKSTRA - FARMACIAS SIMAN SPS

Datos cargados directamente desde:
C:\Users\royum\Downloads\Proyecto_Discretas\Datos_SucursalesSimán.xlsx

ESTRUCTURA DEL PROYECTO
------------------------------------------------------------------------------------------
Cantidad de farmacias por bloque:
  Bloque 1: 12 farmacias
  Bloque 2: 12 farmacias
  Bloque 3: 12 farmacias
  Bloque 4: 12 farmacias
  Bloque 5: 12 farmacias

CANDIDATOS
------------------------------------------------------------------------------------------
Candidato A - Zona Norte
  Ubicación: Sector La 105 Brigada
Candidato B - Zona Sur
  Ubicación: Zona industrial, Sector el Polvorín
Candidato C - Zona Centro
  Ubicación: Barrio Los Andes, 21102 San Pedro Sula, Cortés, Honduras

Metodo de cálculo:
  • Dijkstra utiliza el promedio de ida/vuelta como peso de cada conexion.
  • La ruta se obtiene unicamente mediante las conexiones del grafo.
  • Luego se calculan por separado la distancia de ida, vuelta y promedio.

# Explicación del funcionamiento del proyecto Dijkstra – Farmacias Simán

## 1. Datos del proyecto

El proyecto trabaja con **60 farmacias**, divididas en **5 bloques**:

| Bloque | Cantidad de farmacias |
|---|---:|
| Bloque 1 | 12 |
| Bloque 2 | 12 |
| Bloque 3 | 12 |
| Bloque 4 | 12 |
| Bloque 5 | 12 |
| **Total** | **60** |

Cada farmacia representa un **vértice** del grafo.

Las conexiones entre farmacias representan las **aristas**.

Cada arista tiene un peso asociado a una distancia.

---

## 2. Los tres candidatos

El programa trabaja con tres candidatos:

### Candidato A — Zona Norte
**Ubicación:** Sector La 105 Brigada.

### Candidato B — Zona Sur
**Ubicación:** Zona industrial, Sector el Polvorín.

### Candidato C — Zona Centro
**Ubicación:** Barrio Los Andes.

---

## 3. ¿Qué hace el programa?

El programa toma **un candidato a la vez** y analiza los **5 bloques**.

Por ejemplo:

```text
Candidato A
    │
    ├── Bloque 1
    ├── Bloque 2
    ├── Bloque 3
    ├── Bloque 4
    └── Bloque 5
```

Después se realiza el mismo procedimiento para los candidatos B y C.

---

## 4. ¿Qué sucede al analizar un bloque?

Supongamos que estamos analizando:

```text
Candidato A → Bloque 1
```

El Bloque 1 contiene **12 farmacias**.

Dijkstra calcula las distancias mínimas desde el Candidato A hasta cada una de las 12 farmacias, siguiendo únicamente las conexiones existentes en el grafo.

Conceptualmente:

```text
Candidato A
     │
     ↓
33 Calle Col Guillen
     │
     ├── Simán Luisiana
     ├── VILLA OLIMPICA
     ├── SIMÁN PLAZA VICAY
     ├── Simán Zip San Jose
     └── ...
```

---

## 5. Dijkstra NO realiza 12 recorridos

Cuando la consola muestra:

```text
1. Farmacia A → 8.90 km
2. Farmacia B → 9.93 km
3. Farmacia C → 10.70 km
...
12. Farmacia L → 21.45 km
```

**NO significa que el candidato realizó 12 viajes.**

Significa que Dijkstra calculó las **distancias mínimas desde el candidato hacia las 12 farmacias del bloque**.

Después, el programa ordena esas distancias de menor a mayor.

Por eso aparecen las 12 farmacias.

---

## 6. ¿Por qué aparecen 12 farmacias?

Porque cada bloque tiene 12 farmacias.

El proceso es:

```text
12 farmacias del bloque
        ↓
Dijkstra calcula las distancias mínimas
        ↓
Se obtienen 12 resultados
        ↓
Se ordenan de menor a mayor
```

Por ejemplo:

```text
8.90 km
9.93 km
10.70 km
10.75 km
12.40 km
...
21.45 km
```

La primera posición representa la menor distancia y la posición 12 representa la mayor distancia dentro de ese bloque.

---

## 7. Candidato A analiza los 5 bloques

Para el Candidato A:

```text
Candidato A
    │
    ├── Bloque 1 → 12 farmacias
    ├── Bloque 2 → 12 farmacias
    ├── Bloque 3 → 12 farmacias
    ├── Bloque 4 → 12 farmacias
    └── Bloque 5 → 12 farmacias
```

Por lo tanto:

$$
5 \times 12 = 60
$$

El Candidato A obtiene información sobre las 60 farmacias.

Esto **no significa 60 viajes físicos**. Son 60 distancias mínimas calculadas dentro del grafo.

---

## 8. Candidato B

Se repite exactamente el mismo proceso:

```text
Candidato B
    │
    ├── Bloque 1 → 12 farmacias
    ├── Bloque 2 → 12 farmacias
    ├── Bloque 3 → 12 farmacias
    ├── Bloque 4 → 12 farmacias
    └── Bloque 5 → 12 farmacias
```

Por lo tanto:

$$
5 \times 12 = 60
$$

---

## 9. Candidato C

Finalmente:

```text
Candidato C
    │
    ├── Bloque 1 → 12 farmacias
    ├── Bloque 2 → 12 farmacias
    ├── Bloque 3 → 12 farmacias
    ├── Bloque 4 → 12 farmacias
    └── Bloque 5 → 12 farmacias
```

También:

$$
5 \times 12 = 60
$$

---

## 10. Cantidad total de resultados

Tenemos:

- 3 candidatos
- 5 bloques
- 12 farmacias por bloque

Por lo tanto:

$$
3 \times 5 \times 12 = \boxed{180}
$$

El programa obtiene **180 resultados de distancia candidato → farmacia**.

Esto tampoco significa 180 viajes físicos. Son resultados de las distancias mínimas calculadas mediante el algoritmo.

---

## 11. ¿Qué significa "las 2 farmacias más cercanas"?

Después de calcular las 12 distancias de un bloque, el programa las ordena de menor a mayor.

Ejemplo:

```text
1. Farmacia A → 2.30 km
2. Farmacia B → 3.15 km
3. Farmacia C → 5.00 km
4. Farmacia D → 5.50 km
...
12. Farmacia L → 7.62 km
```

Las dos primeras son las dos farmacias más cercanas dentro de ese bloque:

```text
LAS 2 FARMACIAS MÁS CERCANAS

1. Farmacia A → 2.30 km
2. Farmacia B → 3.15 km
```

---

## 12. ¿Qué significa "farmacia más cercana"?

En este proyecto significa:

> **La farmacia cuya distancia mínima, siguiendo las conexiones del grafo, es menor.**

No se utiliza:

- distancia en línea recta;
- distancia geográfica directa;
- solamente las coordenadas;
- una comparación visual del mapa.

Se utiliza:

$$
\boxed{\text{Dijkstra + pesos del grafo}}
$$

---

## 13. Peso utilizado por Dijkstra

Cada conexión contiene:

- distancia de ida;
- distancia de vuelta.

Por ejemplo:

```text
Farmacia A → Farmacia B

Ida:     2.20 km
Vuelta:  2.40 km
```

El peso utilizado por Dijkstra es el promedio:

$$
Peso = \frac{2.20 + 2.40}{2}
$$

$$
Peso = 2.30\ km
$$

Por lo tanto, Dijkstra utiliza el **promedio de ida y vuelta como peso de cada conexión**.

---

## 14. Cálculo de ida, vuelta y promedio

Después de que Dijkstra encuentra una ruta, el programa calcula por separado la distancia de ida y la distancia de vuelta.

Por ejemplo:

```text
Candidato A
     ↓
Farmacia 1
     ↓
Farmacia 2
     ↓
Farmacia destino
```

Se obtiene:

$$
D_{ida}
$$

$$
D_{vuelta}
$$

Y finalmente:

$$
D_{promedio}
=
\frac{D_{ida}+D_{vuelta}}{2}
$$

Por ejemplo:

```text
Farmacia X

Ida:                 8.80 km
Vuelta:              9.00 km
Promedio:            8.90 km
```

---

## 15. Flujo completo del programa

```text
                    DATOS DEL EXCEL
                           │
                           ↓
                  CONSTRUIR EL GRAFO
                           │
             ┌─────────────┼─────────────┐
             ↓             ↓             ↓
        CANDIDATO A   CANDIDATO B   CANDIDATO C
             │             │             │
             ↓             ↓             ↓
        ┌────────┐    ┌────────┐    ┌────────┐
        │  B1    │    │  B1    │    │  B1    │
        │  B2    │    │  B2    │    │  B2    │
        │  B3    │    │  B3    │    │  B3    │
        │  B4    │    │  B4    │    │  B4    │
        │  B5    │    │  B5    │    │  B5    │
        └────────┘    └────────┘    └────────┘
             │             │             │
             ↓             ↓             ↓
          DIJKSTRA      DIJKSTRA      DIJKSTRA
             │             │             │
             ↓             ↓             ↓
       12 farmacias  12 farmacias  12 farmacias
             │             │             │
             ↓             ↓             ↓
          ORDENAR       ORDENAR       ORDENAR
          menor→mayor   menor→mayor   menor→mayor
             │             │             │
             ↓             ↓             ↓
         2 MÁS         2 MÁS         2 MÁS
        CERCANAS      CERCANAS      CERCANAS
```

---

## 16. ¿Qué se obtiene para cada candidato y bloque?

Para cada combinación de candidato y bloque tendremos:

```text
┌─────────────────────────────────────────┐
│ Candidato A → Bloque 1                  │
├─────────────────────────────────────────┤
│ 12 farmacias ordenadas                  │
│                                         │
│ 1. Farmacia X → XX km                   │
│ 2. Farmacia Y → XX km                   │
│ ...                                     │
│ 12. Farmacia Z → XX km                  │
│                                         │
│ LAS 2 MÁS CERCANAS                       │
│ 1. Farmacia X                           │
│ 2. Farmacia Y                           │
│                                         │
│ Ida                                     │
│ Vuelta                                  │
│ Promedio                                │
│ Ruta encontrada por Dijkstra            │
└─────────────────────────────────────────┘
```

Esto se repite para:

$$
3\ candidatos \times 5\ bloques = \boxed{15\ análisis}
$$

Cada análisis contiene las 12 farmacias correspondientes a su bloque.

---

## 17. Idea principal para recordar

La lógica fundamental del proyecto es:

> **El candidato se conecta al bloque, Dijkstra calcula las distancias mínimas hacia las 12 farmacias del bloque siguiendo el grafo, y posteriormente se ordenan esas 12 distancias para identificar las más cercanas.**

En forma resumida:

$$
\boxed{
\text{Candidato}
\rightarrow
\text{Bloque}
\rightarrow
\text{Dijkstra}
\rightarrow
\text{12 distancias}
\rightarrow
\text{Ordenar}
\rightarrow
\text{2 más cercanas}
}
$$

### Importante

Las 12 farmacias que aparecen en la consola **no representan 12 viajes**. Representan las **12 distancias mínimas calculadas desde el candidato hacia las 12 farmacias de ese bloque**.